# Feature Engineering for the Triple-Barrier Metamodel

This notebook documents and visualises the **feature-engineering layer** built
in `src/stml/metamodel/` for the Imperial/Alken coursework metamodel deliverable.

## What this layer does

The FE layer constructs a **rich, causal feature set** for all 11 futures
instruments (Equity `{es1s, nq1s, fesx1s}`, Energy `{cl1s, ho1s, rb1s, ng1s}`,
Metals `{gc1s, si1s, hg1s, pl1s}`) at each **non-zero-signal trade day**, using
only information $\le t$ (execution is next-day $r_{t+1}$).

Nine feature families are produced:

| Family | Description | Leakage class |
| --- | --- | --- |
| **F1** | Counter-trend / mean-reversion | E |
| **F2** | Volatility & dispersion | E / LI |
| **F5** | Signal-derived (trailing run structure) | E / LI |
| **F6** | Momentum & trend-contrast | E |
| **F7** | Microstructure (volume / open-interest) | E |
| **F8** | Calendar (deterministic sin/cos) | E |
| **F3** | Filtered GMM + Markov regime posteriors | TF |
| **F4** | PCA / KMeans / dense autoencoder (k=4) | TF |
| **F9** | Cross-sectional rank / pair-correlation | E |

## Provenance and causality

**Causal by construction.** Every feature at $t$ uses only data $\le t$:

- **E-class** (engineered, no fit): causality proven by *truncation-invariance*
  — the value at $t$ is identical on `data[:t+1]` and on `data[:T]`.
- **TF-class** (fitted): every GMM / Markov / PCA / KMeans / AE / scaler is
  **fit on the FE-train partition only** (ends `2021-07-01`) and transformed
  with frozen parameters. Causality is proven by a *fit-provenance assertion*
  (training indices $\subseteq$ FE-train) — `test_features_provenance.py`.

**Grading is methodology, not performance** — no performance bar.

## Key references

- Work plan: `.omc/plans/feature-engineering-metamodel-plan.md`
- Graded feature catalog: `reports/feature-catalog.md`
- Persisted matrix: `results/feature_matrix.parquet`
- Redundancy map: `results/feature_redundancy.{json,csv}`
- Scope registry: `results/instrument_scope.json`

## 1. Imports

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

# Public metamodel API — feature logic lives in the package, not here.
from stml.metamodel import (
    FeaturePipeline,
    CATALOG,
    render_catalog,
    build_scope,
    persist_scope,
    InstrumentScope,
)
from stml.metamodel.build_features import compute_redundancy
from stml.io import load_clean_data

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RESULTS = Path("../../results")
REPORTS = Path("../../reports")

## 2. End-to-end build

Load the clean OHLCV + signal panel, fit the full feature pipeline on the
FE-train partition (ends `2021-07-01`), and transform the released window into
the tidy-long feature matrix. This takes approximately 30–60 seconds.

In [ ]:
ohlcv, signals = load_clean_data()
print(f"ohlcv {ohlcv.shape} | signals {signals.shape}")

pipe = FeaturePipeline().fit(ohlcv, signals)
mat = pipe.transform(ohlcv, signals)

print(f"\nFeature matrix shape: {mat.shape}")
print(f"  {mat.shape[0]} rows (non-zero-signal trade days across all instruments)")
print(f"  {mat.shape[1]} columns ({mat.shape[1] - 4} features + 4 meta)")

In [ ]:
print("Partition row counts:")
print(mat["partition"].value_counts().to_string())

print("\nRows per instrument:")
print(mat.groupby("instrument").size().to_string())

print("\nfe_train_end_date attribute:", mat.attrs.get("fe_train_end_date"))

mat.head()

## 3. Per-family feature time-series (si1s)

`si1s` (Silver) is the clean mean-reversion replicator (confirmed passer in the
replication study) — an ideal representative instrument. We plot one or two
features from each of the six engineered families over its full date range.

All features are computed on non-zero-signal trade days using only information
$\le t$ (E-class, proven causal by truncation-invariance).

In [ ]:
INST = "si1s"
si = mat[mat["instrument"] == INST].set_index("date").sort_index()

# One or two features per engineered family
FAMILY_FEATURES = [
    ("F1 — Counter-trend",       ["f1_mr_score_20"],                    "#2166ac"),
    ("F2 — Volatility",          ["f2_vol_20"],                          "#762a83"),
    ("F5 — Signal-derived",      ["f5_trailing_run_length"],             "#1b7837"),
    ("F6 — Momentum contrast",   ["f6_ts_momentum_20"],                  "#d6604d"),
    ("F7 — Microstructure",      ["f7_amihud_20"],                       "#f4a582"),
    ("F8 — Calendar",            ["f8_dow_sin"],                         "#8c510a"),
]

n = len(FAMILY_FEATURES)
fig, axes = plt.subplots(n, 1, figsize=(12, 2.5 * n), sharex=True)

for ax, (title, feats, color) in zip(axes, FAMILY_FEATURES):
    for f in feats:
        ax.plot(si.index, si[f], lw=0.9, color=color, label=f)
    # Shade train / val / test regions
    for part, fc in [("train", "#e0f0ff"), ("val", "#fff3cd"), ("test", "#ffe0e0")]:
        idx = si[si["partition"] == part].index
        if len(idx):
            ax.axvspan(idx[0], idx[-1], alpha=0.18, color=fc, label=part if feats == FAMILY_FEATURES[0][1] else "")
    ax.set_title(f"{title}  ({', '.join(feats)})", fontsize=9)
    ax.legend(fontsize=7, loc="upper right")
    ax.margins(x=0.01)

axes[-1].set_xlabel("date")
fig.suptitle(f"Per-family feature time-series — {INST}  (shading: train / val / test)",
             y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 4. Regime-posterior overlays (si1s)

F3 features are the **filtered (one-sided) GMM and Markov-switching posteriors**
of the high-volatility regime. "Filtered" means the posterior at $t$ uses only
returns $\le t$ — never the smoothed (two-sided) marginals that would use future
observations. The GMM and Markov models are fit on FE-train `(ret, vol)` rows
and their parameters are frozen for the full-series transform (TF-class).

We overlay both posteriors against the rolling 20-day realised volatility
(`f2_vol_20`) on a twin axis and shade the dates where the Markov argmax regime
is "high-vol" ($p_{\text{highvol}} > 0.5$).

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 4))

# Primary axis: regime posteriors
gmm = si["f3_gmm_prob_highvol"].dropna()
mkv = si["f3_markov_prob_highvol"].dropna()

ax1.plot(gmm.index, gmm.values, color="#d6604d", lw=1.1, label="GMM P(high-vol)")
ax1.plot(mkv.index, mkv.values, color="#2166ac", lw=1.1, label="Markov P(high-vol) [filtered]")
ax1.axhline(0.5, color="black", lw=0.7, ls="--", alpha=0.5)
ax1.set_ylabel("P(high-vol regime)")
ax1.set_ylim(-0.05, 1.15)

# Shade high-vol periods (Markov argmax = high-vol)
hv = mkv > 0.5
in_hv = False
start = None
for dt, flag in hv.items():
    if flag and not in_hv:
        start = dt
        in_hv = True
    elif not flag and in_hv:
        ax1.axvspan(start, dt, alpha=0.12, color="#d6604d")
        in_hv = False
if in_hv:
    ax1.axvspan(start, mkv.index[-1], alpha=0.12, color="#d6604d")

# Twin axis: rolling volatility
ax2 = ax1.twinx()
vol = si["f2_vol_20"].dropna()
ax2.plot(vol.index, vol.values, color="#762a83", lw=0.8, alpha=0.6, ls=":",
         label="f2_vol_20 (rolling 20d vol)")
ax2.set_ylabel("Annualised vol (20d)", color="#762a83")
ax2.tick_params(axis="y", labelcolor="#762a83")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc="upper left")

ax1.set_title(
    f"{INST} — F3 filtered regime posteriors (shaded = Markov high-vol; "
    "dotted = rolling vol)",
    fontsize=10,
)
ax1.set_xlabel("date")
ax1.margins(x=0.01)
plt.tight_layout()
plt.show()

print(f"GMM train bundle n_eff_gate: {pipe.scope[INST].n_eff_gate}")
print(f"Markov train index: {pipe._regime[INST].train_index[0].date()} .. "
      f"{pipe._regime[INST].train_index[-1].date()} "
      f"({len(pipe._regime[INST].train_index)} rows)")

## 5. AE vs PCA(k=4) reconstruction MSE per asset class

The dense autoencoder (`n_in → 8 → 4 → 8 → n_in`, MSE, fit on pooled FE-train
within each asset class) is benchmarked against PCA(k=4) on train-block
reconstruction MSE. The AE is **retained as a feature regardless of which is
lower** — there is no performance gate.

Access via `pipe._latent[cls].recon_mse` which holds `{"pca_k4": float, "ae_k4": float}`.

In [ ]:
asset_classes = sorted(pipe._latent.keys())
pca_mse = [pipe._latent[cls].recon_mse["pca_k4"] for cls in asset_classes]
ae_mse  = [pipe._latent[cls].recon_mse["ae_k4"]  for cls in asset_classes]

print("Reconstruction MSE (train block, scaled features):")
for cls, p, a in zip(asset_classes, pca_mse, ae_mse):
    lower = "AE" if a < p else "PCA"
    print(f"  {cls}: PCA(k=4)={p:.5f}  AE(k=4)={a:.5f}  -> lower: {lower}")

x = np.arange(len(asset_classes))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 4))
bars_pca = ax.bar(x - w / 2, pca_mse, width=w, color="#4393c3",
                  edgecolor="black", lw=0.5, label="PCA(k=4)")
bars_ae  = ax.bar(x + w / 2, ae_mse,  width=w, color="#d6604d",
                  edgecolor="black", lw=0.5, label="AE(k=4)")

for bar in list(bars_pca) + list(bars_ae):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f"{bar.get_height():.4f}",
            ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(asset_classes, fontsize=10)
ax.set_ylabel("Train-block reconstruction MSE (scaled)")
ax.set_title(
    "AE(k=4) vs PCA(k=4) reconstruction MSE per asset class\n"
    "(AE retained regardless — no performance gate)"
)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## 6. Feature redundancy — correlation heatmap + dendrogram

Feature correlations are computed via `compute_redundancy` (which reuses
`na_checks.corr_max_info`, pairwise-complete, min_periods=252). Features are
hierarchically clustered on `1 - |corr|` distance (average linkage);
a flat cut at `distance_threshold=0.30` (i.e. `|corr| > 0.70`) groups the
highly-correlated families.

Visually prominent redundant groups:
- **F2 range-vol estimators**: `f2_parkinson_20` / `f2_garman_klass_20` /
  `f2_vol_10` / `f2_vol_20` / `f2_vol_60` — all measure volatility level.
- **F5 run-length pair**: `f5_trailing_run_length` ≈ `f5_days_since_flip`
  (the latter is run-length minus one, by construction).

In [ ]:
corr, clusters, partners = compute_redundancy(mat)

print(f"Feature correlation matrix: {corr.shape}")
print(f"Number of redundancy clusters (|corr|>0.70): {max(clusters.values())}")

# Show the most redundant pairs
top_pairs = sorted(partners.items(), key=lambda x: x[1]["abs_corr"], reverse=True)[:8]
print("\nTop 8 most-correlated pairs:")
for feat, info in top_pairs:
    print(f"  {feat:35s} <-> {info['partner']:35s}  |corr| = {info['abs_corr']:.3f}")

In [ ]:
# Reorder columns by hierarchical cluster for visual grouping
feat_cols = list(corr.columns)
abs_corr_np = corr.abs().to_numpy(dtype=float).copy()
np.fill_diagonal(abs_corr_np, 1.0)
distance = 1.0 - abs_corr_np
distance = (distance + distance.T) / 2.0
np.fill_diagonal(distance, 0.0)
condensed = squareform(distance, checks=False)
Z = linkage(condensed, method="average")

# Get leaf order from dendrogram (suppress plot)
ddata = dendrogram(Z, no_plot=True)
order = ddata["leaves"]
ordered_cols = [feat_cols[i] for i in order]
corr_ordered = corr.loc[ordered_cols, ordered_cols]

fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(
    corr_ordered.abs(),
    cmap="RdYlBu_r",
    vmin=0, vmax=1,
    square=True,
    linewidths=0,
    cbar_kws={"label": "|corr|", "shrink": 0.6},
    ax=ax,
    xticklabels=True,
    yticklabels=True,
)
ax.set_title(
    "Feature correlation heatmap (|corr|, hierarchically ordered)\n"
    "Bright red = high redundancy (|corr|>0.70)",
    fontsize=11,
)
ax.tick_params(axis="x", labelsize=6, rotation=90)
ax.tick_params(axis="y", labelsize=6, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
dendrogram(
    Z,
    labels=feat_cols,
    leaf_rotation=90,
    leaf_font_size=7,
    color_threshold=0.30,
    ax=ax,
)
ax.axhline(0.30, color="#d6604d", lw=1.1, ls="--",
           label="|corr|>0.70 cut (distance=0.30)")
ax.set_title(
    "Feature redundancy dendrogram (1 - |corr| distance, average linkage)\n"
    "Clusters below the red dashed line have |corr|>0.70",
    fontsize=10,
)
ax.set_ylabel("1 - |corr|")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Scope registry (D5)

The `InstrumentScope` registry encodes the per-model-type fitting-scope decision
(plan §D5): regime models (F3 GMM + Markov) are **per-instrument** (fit on each
instrument's own dense daily return series, ~5,600–7,900 returns through
2021-07-01); latent models (F4 PCA / KMeans / AE) are **pooled within asset
class** (EQ 1083, EN 668, ME 1174 FE-train rows).

Instruments with `n_eff_gate < 10` are flagged **low-power**: `{cl1s, ho1s, ng1s}`.

In [ ]:
scope_df = pd.DataFrame([
    {
        "instrument": sc.instrument,
        "asset_class": sc.asset_class,
        "n_eff_gate": sc.n_eff_gate,
        "fit_scope_regime": sc.fit_scope_regime,
        "fit_scope_latent": sc.fit_scope_latent,
        "low_power": sc.low_power,
    }
    for sc in pipe.scope.values()
]).sort_values(["asset_class", "instrument"]).reset_index(drop=True)

print("D5 InstrumentScope registry:")
print(scope_df.to_string(index=False))

low_power = scope_df[scope_df["low_power"]]["instrument"].tolist()
print(f"\nLow-power instruments (n_eff_gate < 10): {low_power}")
print("  -> pooled at class level for downstream verdicts; no standalone claims.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))
colors = ["#d6604d" if lp else "#4393c3"
          for lp in scope_df["low_power"]]
bars = ax.bar(scope_df["instrument"], scope_df["n_eff_gate"],
              color=colors, edgecolor="black", lw=0.5)
ax.axhline(10, color="black", lw=1, ls="--", label="FLOOR = 10")
for bar, cls in zip(bars, scope_df["asset_class"]):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            cls, ha="center", va="bottom", fontsize=7)
ax.set_ylabel("n_eff_gate (post-embargo val)")
ax.set_title(
    "Gateable n_eff per instrument (red = low-power, below FLOOR=10)\n"
    "Labels show asset class (EQ / EN / ME)"
)
ax.legend(fontsize=8)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 8. Feature catalog summary

The graded artifact is `reports/feature-catalog.md` (rendered by
`stml.metamodel.catalog.render_catalog`). Here we display the catalog size,
the leakage-class breakdown, and a sample of `FeatureSpec` entries grouped by
leakage class.

In [ ]:
print(f"CATALOG size: {len(CATALOG)} FeatureSpec entries")
print()

by_class = {"E": [], "TF": [], "LI": []}
for name, spec in CATALOG.items():
    by_class[spec.leakage_class].append(spec)

for lc, specs in by_class.items():
    print(f"Leakage class {lc}: {len(specs)} features")

print()
# Sample: two entries per leakage class
rows = []
for lc in ["E", "TF", "LI"]:
    for spec in by_class[lc][:2]:
        rows.append({
            "leakage_class": spec.leakage_class,
            "family": spec.family,
            "name": spec.name,
            "lookback": spec.lookback,
            "what_it_captures": spec.what_it_captures[:70] + "...",
        })

sample_df = pd.DataFrame(rows)
print("Sample FeatureSpec entries (2 per leakage class):")
print(sample_df.to_string(index=False))
print()
print(f"Full graded catalog: {REPORTS / 'feature-catalog.md'}")

## 9. FE → model handoff contract

The feature matrix embeds the full provenance needed for the deferred
triple-barrier metamodel phase:

| Contract item | Value |
| --- | --- |
| `fe_train_end_date` | `2021-07-01` (inclusive FE-train boundary) |
| Train partition | 2020-01-03 .. 2021-07-01 (2,925 rows) |
| Val partition | 2021-07-02 .. 2021-12-30 (1,041 rows) |
| Test partition | 2021-12-31 .. 2022-06-30 (1,018 rows) |
| Canonical matrix | `results/feature_matrix.parquet` |
| Label-interface subset | `f2_vol_20` (barrier sigma) + `f5_trailing_run_length` (run membership) |
| Embargo sizing | `run_length_p90` per instrument (from `splits`, reserved for CV only) |

**The downstream model phase must not retrain fitted features (F3/F4)
on observations after `fe_train_end_date`.** It may either:
1. Load the persisted `feature_matrix.parquet` and treat the features as
   causal-forward (the normal path); or
2. Re-run `FeaturePipeline().fit(ohlcv, signals)` with the same
   `fe_train_end="2021-07-01"` boundary (deterministic rebuild).

The `partition` column in the matrix encodes this contract directly, and
`test_features_provenance.py` asserts that it is honoured in the persisted
artifacts.

In [ ]:
# Provenance summary from the fitted pipeline
provenance_path = RESULTS / "feature_matrix_provenance.json"
if provenance_path.exists():
    prov = json.loads(provenance_path.read_text())
    print("feature_matrix_provenance.json:")
    for k, v in prov.items():
        if k not in ("regime_train_index", "latent_train_index"):
            print(f"  {k}: {v}")
    print("\nLatent train index counts + recon_mse per class:")
    for cls, info in prov.get("latent_train_index", {}).items():
        print(f"  {cls}: count={info['count']}  recon_mse={info['recon_mse']}")
else:
    # Compute from the fitted pipeline directly
    print("fe_train_end_date:", pipe.fe_train_end)
    print("train:", pipe._train_dates[0].date(), "..", pipe._train_dates[-1].date())
    print("val:  ", pipe._val_dates[0].date(),   "..", pipe._val_dates[-1].date())
    print("test: ", pipe._test_dates[0].date(),  "..", pipe._test_dates[-1].date())
    print()
    for cls, bundle in pipe._latent.items():
        print(f"  {cls}: latent_train_count={len(bundle.train_index)}  recon_mse={bundle.recon_mse}")

print("\nHandoff complete. Feature matrix with partition provenance is ready for")
print("the deferred triple-barrier label + metamodel phase.")